### Ray Tracing & Graphical Pipeline

#### Objective: 
1. $\text{Compare between the basic algorithms of the Graphical pipeline and Ray Tracing}$
2. $\text{Summarise the factors leading to their basic complexities and conclude the complexities}$
3. $\text{What we can't optimise and why and how we can optimize and conclude optimized comlpexity}$

##### $\text{Assumptions:}$

$\text{We wish to create an image of HxW from a 3D scene, where:}$

1. $\text{There are M objects in the scene}$

2. $\text{With L light sources}$

3. $\text{In total the objects are defined using T triangles}$







### 📊 Ray Tracing vs Graphical Pipeline (Basic)

| **Category**             | **Ray Tracing**                                                                                                                                                                                                                                                  | **Graphical Pipeline**                                                                                                                                                                                                                                                                          |
|--------------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **Algorithm Explanation** | Shoots a ray from the camera through each pixel into the 3D scene. If it hits an object, it spawns: <br> 1. **Shadow Ray** toward light sources. <br> 2. **Reflected Ray** using surface normals. <br> 3. **Refracted Ray** bending based on refractive index. <br> Rays are traced recursively up to depth $begin:math:text$ R $end:math:text$, collecting colors back to the pixel. | Operates on **polygonal geometry** (usually triangles): <br> 1. Lighting computed at **vertices** (Phong/Gouraud). <br> 2. Vertices **projected** from 3D to 2D. <br> 3. **Rasterization** interpolates color/depth. <br> 4. **Z-buffer** stores the closest depth per pixel for visibility. |
| **Advantages**           | - Physically accurate lighting/shadows. <br> - Simulates reflection, refraction, and caustics. <br> - Produces highly **realistic images**.                                                                                                                        | - Extremely **fast** rendering. <br> - Efficient on the **GPU**. <br> - Ideal for **real-time** applications. <br> - Mature APIs (OpenGL, DirectX).                                                                                                                                               |
| **Disadvantages**        | - **Slow** and computationally expensive. <br> - Not practical for real-time without hardware acceleration (e.g., RTX). <br> - High memory and CPU usage per ray.                                                                                                   | - Relies on **approximations** (e.g., fake reflections). <br> - Limited realism. <br> - Lacks natural global illumination/transparency. <br> - Can produce aliasing/shading artifacts.                                                                                                            |
| **Use Case**             | - **Pre-rendered** visual effects (films). <br> - Architectural rendering. <br> - Scientific visualization. <br> - **Offline** rendering where quality is more important than speed.                                                                                | - **Video games**, simulations, AR/VR. <br> - Real-time user interfaces. <br> - Applications needing fast feedback over photorealism.                                                                                                                                                            |

### Complexity Analysis

| **Aspect**                     | **Ray Tracing**                                                                                                             | **Graphical Rendering Pipeline**                                                                                                   |
|-------------------------------|-----------------------------------------------------------------------------------------------------------------------------|------------------------------------------------------------------------------------------------------------------------------------|
| **Assumptions**               | - $L$: Number of light sources  <br> - $H \times W$: Image resolution (pixels) <br> - $T$: Number of triangles <br> - $R$: Recursion depth | - $L$: Number of light sources  <br> - $H \times W$: Image resolution (pixels) <br> - $T$: Number of triangles                    |
| **Computation Per Ray**       | For a **single ray**: check **intersection** with all triangles and light sources: $O(T + L)$                              | For a **single triangle**: compute lighting and interpolate values over its surface: $O(T + L)$ (dependent on lights and geometry) |
| **Rays per Pixel**            | $1$ initial ray per pixel → splits into $\approx 2^R - 1$ rays recursively                                                  | Each triangle may affect multiple pixels depending on projection and rasterization                                                 |
| **Total Work**                | $O\big((HW)(2^R - 1)(T + L)\big)$                                                                                           | $O\big(HW \cdot (T + L)\big)$                                                                                                      |
| **Intuition Behind Cost**     | Each pixel spawns a **ray tree** of depth $R$; each ray checks **all triangles and lights**                                | Each triangle is **projected**, and then pixel colors are interpolated and shaded using **local lighting**                         |
| **Bottleneck**                | - Recursion depth $R$ <br> - Triangle count $T$ <br> - Light count $L$                                                       | - Triangle count $T$ <br> - Pixel fill rate $HW$                                                                                   |
| **Use Case Impact**           | Very expensive without acceleration; used in offline rendering                                                              | Efficient for real-time rendering due to rasterization and interpolation                                                           |



### 📊 Ray Tracing vs Graphical Pipeline (Optimizing)

| **Aspect**                 | **Ray Tracing**                                                                                                                                                                                                                                 | **Graphical Rendering Pipeline**                                                                                                                                                                                              |
|---------------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **Fixed Parameters**      | - $H \times W$: Fixed image resolution <br> - $L$: Number of light sources (application-dependent) <br> - $R$: Recursion depth (application-dependent)                                                                                         | - $H \times W$: Fixed image resolution <br> - $L$: Number of light sources (application-dependent)                                                                                                                            |
| **Cost Bottleneck**       | Each ray must test **intersection with all $T$ triangles** → very expensive due to exhaustive checks.                                                                                                                                           | Each triangle must be **projected and rasterized**, which is expensive if $T$ is large.                                                                                                                                        |
| **Optimization Technique**| **Bounding Volume Hierarchy (BVH)**: <br> - Organizes the scene into nested bounding boxes. <br> - Intersections tested **hierarchically**: check boxes first, descend only when needed. <br> - Reduces triangle checks drastically.            | **Multi-Resolution / Level of Detail (LoD)**: <br> - Store multiple mesh versions per model. <br> - Use fewer triangles when: <br>  • Object is far from camera <br>  • Object is poorly lit. <br> - Reduces triangle workload. |
| **Optimized Complexity**  | Instead of checking all $T$ triangles per ray: <br> **Reduced to** $O(\log T)$ via BVH. <br> ⇒ Total: $O\big((HW)(2^R - 1)(\log T + L)\big)$                                                                                                    | If LoD reduces triangle count to $T' \ll T$, then total cost becomes: <br> **$O\big(HW \cdot (T' + L)\big)$**                                                                                                                  |
| **When Effective**        | BVH works best when: <br> - Scene is spatially structured <br> - Triangle groups form tight bounding boxes <br> - Rays mostly intersect small portions of the scene                                                                              | LoD is effective when: <br> - Objects are distant or minimally visible <br> - Triangle detail is visually unnecessary                                                                                                          |
| **Trade-Offs**            | - Added cost to build and maintain BVH structure <br> - Scene must be updated if objects move                                                                                                             | - Must store and switch between multiple mesh versions <br> - May cause visible detail loss if LoD selection is too aggressive                                                                                                |